# D'QUILANE Global — Multilingual + GitHub Pages Ready

**Versi balita multibahasa** + export ke JSON biar bisa jalan di GitHub Pages (browser pure JS).

## Workflow:
1. Train di Kaggle (cell 1-5)
2. Export model ke JSON (cell 6)
3. Download file JSON
4. Upload ke GitHub repo
5. Buka di GitHub Pages → web tester jalan tanpa server!

In [ ]:
# ===== CELL 1: INSTALL =====
import subprocess, sys
for pkg in ['torch', 'gradio']:
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
import torch, gradio, os, math, json, time
import torch.nn as nn
import torch.nn.functional as F
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}  Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ===== CELL 2: MULTILINGUAL CORPUS =====
# Balita level: belajar beberapa kalimat dari berbagai bahasa.
# Tambah/tambah kalimat sesuai keinginan. Tiap bahasa 5-10 kalimat.

TEXT = '''Indonesia adalah negara kepulauan.
Ibu kota Indonesia adalah Jakarta.
Bahasa Indonesia adalah bahasa resmi.
Penduduk Indonesia sangat beragam.
Indonesia kaya akan sumber daya alam.
Mari belajar bahasa Indonesia.
Anak muda harapan masa depan.
Belajar setiap hari bikin hidup bermakna.
Kerja keras kunci pencapaian tujuan.
Indonesia akan terus maju dan berkembang.
English is a global language.
The world is connected by technology.
Computers changed how humans communicate.
Artificial intelligence is a global trend.
Large language models are popular now.
Children learn languages quickly.
Education is the key to the future.
Hard work leads to success.
Knowledge is power and wisdom.
Every day is a chance to learn.
El espanol es un idioma hermoso.
La familia es muy importante.
El mundo es grande y diverso.
La educacion es la clave del futuro.
El trabajo duro trae exito.
Cada dia es una nueva oportunidad.
Los ninos aprenden rapidamente.
La tecnologia avanza rapidamente.
El conocimiento es poder.
La inteligencia artificial es el futuro.
Le francais est une belle langue.
La famille est tres importante.
Le monde est grand et diversifie.
L education est la cle du futur.
Le travail dur mene au succes.
Chaque jour est une chance d apprendre.
Les enfants apprennent vite.
La technologie avance rapidement.
Le savoir est le pouvoir.
Deutsch ist eine wichtige Sprache.
Die Familie ist sehr wichtig.
Die Welt ist gross und vielfaeltig.
Bildung ist der Schluessel zur Zukunft.
Harte Arbeit fuehrt zum Erfolg.
Jeder Tag ist eine neue Chance.
Kinder lernen schnell.
Die Technik schreitet voran.
Wissen ist Macht.
Il italiano e una bella lingua.
La famiglia e molto importante.
Il mondo e grande e diversificato.
L istruzione e la chiave del futuro.
Il lavoro duro porta al successo.
Ogni giorno e una nuova opportunita.
I bambini imparano velocemente.
La tecnologia avanza rapidamente.
La conoscenza e potere.
Portugues e uma lingua bonita.
A familia e muito importante.
O mundo e grande e diversificado.
A educacao e a chave do futuro.
O trabalho duro leva ao sucesso.
Cada dia e uma nova oportunidade.
As criancas aprendem rapidamente.
A tecnologia avanca rapidamente.
O conhecimento e poder.
Russian is also spoken by many.
Russian uses the Cyrillic alphabet.
Languages connect people.
Every language has unique words.
Some words are difficult to translate.
Language learning is fun.
Reading expands the mind.
Writing clarifies thoughts.
Speaking builds confidence.
Listening is the first skill.
Chinese is the most spoken language.
Japanese has three writing systems.
Korean uses Hangul script.
Arabic is written right to left.
Hindi uses the Devanagari script.
Greek letters are used in science.
Latin is the root of many languages.
Sanskrit is an ancient language.
Translation connects cultures.
Poetry is beautiful in any language.
Music is the universal language.
Mathematics is also a language.
Code is the language of computers.
Python is popular for AI.
JavaScript runs in the browser.
HTML structures web pages.
CSS styles the appearance.
GitHub hosts code repositories.
Open source empowers everyone.
Sharing knowledge is noble.
Collaboration builds better things.
The future is multilingual.
'''

In [ ]:
# ===== CELL 3: CONFIG + TOKENIZER =====
CONFIG = {
    'vocab_size': 0,  # akan di-set otomatis
    'd_model':    96,
    'n_heads':    4,
    'd_ff':       384,
    'n_layers':   6,    # virtual depth
    'max_seq':    64,
    'epochs':     2000,
    'batch_size': 16,
    'lr':         3e-4,
}

# Char-level tokenizer (handles ANY unicode character)
chars = sorted(set(TEXT))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
stoi['<pad>'] = len(stoi)
itos[len(itos)] = '<pad>'
PAD_ID = stoi['<pad>']
VOCAB = len(stoi)
CONFIG['vocab_size'] = VOCAB

def encode(s):
    return [stoi[c] for c in s if c in stoi]
def decode(ids):
    return ''.join(itos.get(int(i), '?') for i in ids if i != PAD_ID)

data = torch.tensor(encode(TEXT), dtype=torch.long)
print(f'Vocab size: {VOCAB} karakter (multilingual: Latin, Cyrillic, CJK, etc.)')
print(f'Total tokens: {len(data)}')
print(f'Sample chars: {[itos[i] for i in range(min(20, VOCAB))]}'  )

def get_batch():
    idx = torch.randint(0, len(data) - CONFIG['max_seq'] - 1, (CONFIG['batch_size'],))
    x = torch.stack([data[i:i+CONFIG['max_seq']] for i in idx])
    y = torch.stack([data[i+1:i+1+CONFIG['max_seq']] for i in idx])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
# ===== CELL 4: MODEL D'QUILANE (sama dengan versi balita) =====
class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dh)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class FFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.val  = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.val(x))

class Block(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn  = Attention(d, h)
        self.norm2 = nn.RMSNorm(d)
        self.ffn   = FFN(d, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DQuilane(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq'], cfg['d_model'])
        self.block = Block(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
        self.norm_f = nn.RMSNorm(cfg['d_model'])
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for _ in range(self.cfg['n_layers']):
            x = self.block(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                    targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=50, temp=0.8, top_k=10):
        self.eval()
        for _ in range(max_new):
            x = idx if idx.size(1) <= self.cfg['max_seq'] else idx[:, -self.cfg['max_seq']:]
            logits, _ = self(x)
            logits = logits[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = DQuilane(CONFIG).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f"D'QUILANE Global lahir! params: {n:,} (~{n*4/1024:.1f} KB)")
print(f'Virtual depth: {CONFIG["n_layers"]} (effective: {n * CONFIG["n_layers"]:,})')

In [ ]:
# ===== CELL 5: TRAINING =====
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])
model.train()
t0 = time.time()
for epoch in range(CONFIG['epochs']):
    xb, yb = get_batch()
    opt.zero_grad()
    _, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    if epoch % 200 == 0 or epoch == CONFIG['epochs'] - 1:
        print(f'epoch {epoch+1:4d}/{CONFIG["epochs"]}  loss={loss.item():.4f}  t={time.time()-t0:.1f}s')
print(f'\nDone in {time.time()-t0:.1f}s. Final loss: {loss.item():.4f}')
model.eval()

In [ ]:
# ===== CELL 6: EXPORT MODEL KE JSON (UTK GITHUB PAGES) =====
# Konversi bobot PyTorch ke JSON yang bisa di-load di browser pakai JavaScript

import json

def tensor_to_list(t):
    """Convert torch tensor ke nested list (JSON-safe)."""
    return t.detach().cpu().tolist()

# ambil bobot dari model
weights = {
    'config': CONFIG,
    'stoi': stoi,                # char -> id
    'itos': {str(k): v for k, v in itos.items()},  # id (as string) -> char
    'weights': {
        # Embeddings
        'tok_emb_weight': tensor_to_list(model.tok_emb.weight),
        'pos_emb_weight': tensor_to_list(model.pos_emb.weight),
        # Block (shared)
        'norm1_weight': tensor_to_list(model.block.norm1.weight),
        'attn_qkv_weight': tensor_to_list(model.block.attn.qkv.weight),
        'attn_out_weight': tensor_to_list(model.block.attn.out.weight),
        'norm2_weight': tensor_to_list(model.block.norm2.weight),
        'ffn_gate_weight': tensor_to_list(model.block.ffn.gate.weight),
        'ffn_val_weight': tensor_to_list(model.block.ffn.val.weight),
        'ffn_down_weight': tensor_to_list(model.block.ffn.down.weight),
        # Final
        'norm_f_weight': tensor_to_list(model.norm_f.weight),
        # head tied ke tok_emb, gak perlu export terpisah
    }
}

os.makedirs('output', exist_ok=True)
JSON_PATH = 'output/dquilane_model.json'
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(weights, f, ensure_ascii=False)

size_kb = os.path.getsize(JSON_PATH) / 1024
print(f'JSON exported: {JSON_PATH} ({size_kb:.1f} KB)')
print(f'\nDownload file ini di tab Output, taruh di folder `web/` repo GitHub Pages kamu.')

# test generate
for p in ['Indonesia', 'English', 'La familia', 'Le francais']:
    ids = torch.tensor([encode(p)], dtype=torch.long, device=DEVICE)
    out = model.generate(ids, max_new=80, temp=0.7, top_k=10)
    print(f'\n[{p}] -> {decode(out[0].tolist())}')

In [ ]:
# ===== CELL 7 (OPTIONAL): GRADIO WEB TESTER (kalau mau coba dulu di Kaggle) =====
# Skip kalau mau langsung ke GitHub Pages

def generate_gradio(prompt, max_new, temp, top_k):
    ids = encode(prompt)
    if not ids:
        return '[prompt gak ada karakter yang dikenal]'
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new=int(max_new), temp=float(temp), top_k=int(top_k))
    return decode(out[0].tolist())

with gr.Blocks(title="D'QUILANE Global Tester") as demo:
    gr.Markdown("""
    # D'QUILANE Global v0.3
    Multilingual LLM (balita level) — bisa ngomong bahasa campuran.
    """)
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Prompt', value='Indonesia', lines=2)
            max_new = gr.Slider(20, 300, value=80, step=10, label='Max tokens')
            temp = gr.Slider(0.1, 2.0, value=0.7, step=0.1, label='Temperature')
            top_k = gr.Slider(0, 50, value=10, step=1, label='Top-k')
            btn = gr.Button('Generate', variant='primary')
        with gr.Column():
            output = gr.Textbox(label='Output', lines=12)
    btn.click(generate_gradio, [prompt, max_new, temp, top_k], output)
    gr.Examples([['Indonesia'], ['English is'], ['La famille'], ['Le francais']],
                inputs=prompt)

demo.launch(share=True, debug=False)